In [20]:
import anthropic
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display
import pandas as pd
import numpy as np
load_dotenv()

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_KEY"))

print("Client ready.")

Client ready.


In [21]:
USE_CACHED = True  # Set to False to re-run API calls and overwrite saved results

In [22]:
import pickle

with open("../data/train_test_data.pkl", "rb") as f:
    data =pickle.load(f)

import pandas as pd
comparison_df = pd.read_csv("../outputs/model_comparison.csv", index_col= "Model")

print(comparison_df)

                              model       rmse       mae  directional_accuracy
Model                                                                         
LinearRegression  Linear Regression   2.067377  1.578433                 68.88
LSTM                           LSTM   7.236123  5.971854                 52.13
RandomForest           RandomForest  10.599141  8.051550                 63.90
XGBoost                     XGBoost  11.012762  8.554283                 67.22


In [23]:
import os
print(os.getcwd())

c:\Users\david\Downloads\Projects\Stock Machine Learning & AI\stock-forecast-ml-pipeline\notebooks


In [24]:
import os
for f in os.listdir("../data"):
    print(f)

train_test_data.pkl


In [25]:
def get_model_commentary(comparison_df):
    """
    Input: comparsion dataframe
    output: analyst-style commentary from Claude
    """
    #dataframe to string
    metrics_str = comparison_df.to_string()

    #THE PROMPT ENGINEERING

    prompt = f"""
    You are a quantitative analyst reviewing the results of a stock price forecasting experiment.
    Four models are trained on 5 years of AAPL historical data and evaluated on three metrics:
    - RMSE (Root Mean Square Error) - lower is better
    - MAE (Mean Average Value) - lower is better
    - Directional Accuracy - higher is better, 50% is a coin flip
    - For RMSE and MAE, these measure the difference between the actual and predicted stock prices

    Here are the results:
    {metrics_str}

    Write a concise analyst-style commentary (3-4 sentences) that:
    1. Identifies the best performing model and why
    2. Notes any interesting patterns between the models
    3. Comments on what the directional accuracy results mean practically
    Keep the tone professional but accessible to a non-technical audience. 
    Key ideas should be clearly conveyed.

"""
    message = client.messages.create(
        model= "claude-opus-4-5",
        max_tokens= 500,
        messages=[
            {"role" : "user", "content": prompt}
        ]
    )
    return(message.content[0].text)


In [26]:
if USE_CACHED:
    with open("../outputs/overall_commentary.json", "r") as f:
        commentary = json.load(f)["overall_commentary"]
    print("Loaded overall commentary from cache.")
else:
    commentary = get_model_commentary(comparison_df)

Loaded overall commentary from cache.


In [27]:
display(Markdown(commentary))


## Model Performance Analysis

**Linear Regression emerges as the clear winner**, delivering the lowest prediction errors (RMSE: 2.07, MAE: 1.58) by a substantial margin—nearly four times more accurate than the next best model. Interestingly, the more complex machine learning models (LSTM, Random Forest, XGBoost) significantly underperformed the simpler Linear Regression approach, suggesting that AAPL's price movements over this period may follow relatively straightforward trends that don't require sophisticated pattern recognition.

From a practical trading perspective, the directional accuracy results are encouraging: Linear Regression correctly predicted whether the stock would move up or down nearly 69% of the time—well above the 50% random baseline. This means an investor using this model's directional signals would have been right roughly 7 out of 10 times, providing a meaningful edge for timing decisions, though the absolute price predictions should still be used cautiously given inherent market uncertainties.

In [28]:
def get_per_model_commentary(model_name, metrics):
    """
    Input: model names and their metrics (MAE, RMSE and Direcrtional accuracy) as dict
    Output: analyst-style commentary for each model
    """
    prompt = f"""
    You are a seasoned quantitative analyst wrting a brief performace review for a single model
    in a stock price forecasting experiment

    Model:{model_name}
    RMSE : {metrics['rmse']} (lower is better, measures average dollar error)
    MSE : {metrics['mae']} (lower is better, average absolute dollar error)
    Directional Accuracy : {metrics['directional_accuracy']} ( higher is better and above 50% beats coin flip)

    Context: This model was one of four tested on 5 years of AAPL data.
    The best performing model achieved an RMSE of 2.07, MAE of 1.58, and 68.88% directional accuracy.

    Write 2-3 sentences that:
    1. Summarize how this model perfomed
    2. Compare is to the benchmarks above
    3. Give one practical insight about what these numbers mean

    Be direct and professional. No bullet points23, just prose
"""
    
    message = client.messages.create(
        model = 'claude-opus-4-5',
        max_tokens= 500,
        messages= [
            {"role" : "user", "content" : prompt}
        ]
    )
    return(message.content[0].text)

In [29]:
if USE_CACHED:
    with open("../outputs/per_model_commentary.json", "r") as f:
        per_model_commentary = json.load(f)
    print("Loaded per-model commentary from cache.")
else:
    per_model_commentary = {}

    for model_name, row in comparison_df.iterrows():
        print(f'Genterating commentary for {model_name}..')

        metrics = {
            "rmse" : row["rmse"],
            "mae" : row["mae"],
            "directional_accuracy" : row["directional_accuracy"]
        }
        model_commentary = get_per_model_commentary(model_name, metrics)
        per_model_commentary[model_name] = model_commentary
        print(f"Done.\n")

    print(f'All commentaries generated')

Loaded per-model commentary from cache.


In [30]:
print(per_model_commentary)

{'LinearRegression': '**Performance Review: Linear Regression Model**\n\nThe Linear Regression model delivered strong performance on the AAPL forecasting task, achieving an RMSE of $2.07, MAE of $1.58, and directional accuracy of 68.88%, effectively matching the best benchmarks across all three metrics. This indicates that despite its relative simplicity, the model captures the underlying price dynamics as effectively as more complex alternatives tested in this experiment. From a practical standpoint, the 68.88% directional accuracy suggests the model correctly predicts whether AAPL will move up or down roughly 7 out of 10 trading days, providing meaningful edge over random chance for directional trading strategies.', 'LSTM': '**Performance Review: LSTM Model**\n\nThe LSTM model delivered underwhelming results with an RMSE of $8.02 and directional accuracy of 54.03%, placing it well below the top performer which achieved an RMSE of $2.07 and 68.88% directional accuracy. While the model

In [31]:
with open("../outputs/per_model_commentary.json","w") as f:
    json.dump(per_model_commentary, f , indent= 4)

print("Per-model commentary saved to outputs/per_model_commentary.json")

overall ={"overall_commentary": commentary}

with open("../outputs/overall_commentary.json", "w") as f:
    json.dump(overall, f, indent= 4)

print("Overall commentary of the models saved to outputs/overall_commentary.json")

Per-model commentary saved to outputs/per_model_commentary.json
Overall commentary of the models saved to outputs/overall_commentary.json


In [32]:
with open("../outputs/per_model_commentary.json", "r") as f:
    saved = json.load(f)

for model, text in saved.items():
    print(f"\n {'='*50}")
    print(f"Model: {model}")
    print(f"f{'='*50}")
    print(text)


Model: LinearRegression
f==================================================
**Performance Review: Linear Regression Model**

The Linear Regression model delivered strong performance on the AAPL forecasting task, achieving an RMSE of $2.07, MAE of $1.58, and directional accuracy of 68.88%, effectively matching the best benchmarks across all three metrics. This indicates that despite its relative simplicity, the model captures the underlying price dynamics as effectively as more complex alternatives tested in this experiment. From a practical standpoint, the 68.88% directional accuracy suggests the model correctly predicts whether AAPL will move up or down roughly 7 out of 10 trading days, providing meaningful edge over random chance for directional trading strategies.

Model: LSTM
f==================================================
**Performance Review: LSTM Model**

The LSTM model delivered underwhelming results with an RMSE of $8.02 and directional accuracy of 54.03%, placing it well

In [33]:


def detect_divergences(y_test, y_pred, dates, threshold_pct=0.02):
    """
    Flags dates where predicted vs actual price diverges 
    beyond a percentage threshold.
    
    threshold_pct=0.02 means flag anything over 2% off
    """
    
    y_test_arr = np.array(y_test)
    y_pred_arr = np.array(y_pred)
    
    # Calculate percentage divergence for each prediction
    pct_divergence = np.abs(y_test_arr - y_pred_arr) / y_test_arr
    
    # Flag anything beyond the threshold
    divergence_mask = pct_divergence > threshold_pct
    
    # Build a DataFrame of flagged dates
    divergence_df = pd.DataFrame({
        "date":           dates[divergence_mask],
        "actual":         y_test_arr[divergence_mask].round(2),
        "predicted":      y_pred_arr[divergence_mask].round(2),
        "pct_divergence": (pct_divergence[divergence_mask] * 100).round(2)
    })
    
    divergence_df = divergence_df.sort_values("pct_divergence", ascending=False)
    divergence_df = divergence_df.reset_index(drop=True)
    
    return divergence_df

In [ ]:
dates = pd.Series(data['y_test'].index)

divergence_df = detect_divergences(
    y_test= data['y_test'],
    y_pred= = data['Linea']
)

SyntaxError: invalid syntax (1658203809.py, line 5)

{'x_trained_scaled': array([[0.        , 0.        , 0.        , ..., 0.53558519, 0.47120057,
         0.47819736],
        [0.0029668 , 0.00138445, 0.00106257, ..., 0.5625297 , 0.52735347,
         0.56009356],
        [0.00693618, 0.00288008, 0.00277145, ..., 0.57014378, 0.65721762,
         0.53802901],
        ...,
        [0.62812168, 0.68263883, 0.76425162, ..., 0.53424445, 0.36723742,
         0.50652476],
        [0.63187899, 0.67953138, 0.76091193, ..., 0.53572412, 0.55930058,
         0.46192851],
        [0.64068124, 0.67556016, 0.75792395, ..., 0.60276096, 0.47510449,
         0.39428297]], shape=(967, 9)),
 'x_test_scaled': array([[0.64954679, 0.67172155, 0.7555488 , ..., 0.51537514, 0.6658794 ,
         0.6317892 ],
        [0.66107205, 0.66911589, 0.75420487, ..., 0.55851505, 0.53424445,
         0.52772436],
        [0.66697189, 0.66865353, 0.75368248, ..., 0.55303072, 0.53572412,
         0.36723742],
        ...,
        [1.12720384, 1.14836814, 1.11557057, ..., 0.506